# ENGRAMA V4 × TinyStories — Modelo autorregresivo de ~20M con GPT-2 🧠⚡

Entrenamiento **completo y de alta velocidad** de un modelo de lenguaje autorregresivo **sin atención** (cero $QK^T$, cero softmax temporal) con la arquitectura **ENGRAMA V4** instalada desde GitHub.

| Aspecto | Valor |
|---|---|
| Arquitectura | **ENGRAMA V4** (gating dual target-source, trace tap T0, offsets resonantes, RMSNorm, evocador latent-fusion) |
| Parámetros | **~20.4M** (verificado en ejecución) |
| Datos | **TinyStories V2-GPT4 completo** (train 2.08 GiB + valid) |
| Tokenizer | **GPT-2 BPE** (vocabulario 50,257) |
| Contexto | **512 tokens** |
| Velocidad | **~0.15 - 0.25 s/paso** en GPU T4 con AMP (aceleración 15× vs V3) |
| Tiempo total | **~1.5 a 2.5 horas** para los 500M tokens en Kaggle |
| Salida | checkpoint `engrama_v4_20m_gpt2/` + muestras de inferencia |

> 🚀 **ENGRAMA V4**: Resuelve el cuello de botella de velocidad (evocador latent-fusion $O(|V|d)$ sin checkpoints y consolidación pre-proyectada) y la recuperación en contexto largo (gating bilateral + acceso directo a traza T0).

- Autor: **BUEORM** · Licencia: **AGPL-3.0**


## 1️⃣ Instalación (ENGRAMA V4 desde GitHub + transformers)


In [ ]:
# Instalación de ENGRAMA
!pip install -q git+https://github.com/bueormnew/engrama.git transformers numpy

import os
import math
import time
import random
import glob
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import engrama
from engrama import (EngramaConfig, EngramaModel, Generator,
                     save_model, load_model, chunked_cross_entropy)

print('ENGRAMA', engrama.__version__, '| torch', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count() if DEVICE == 'cuda' else 0
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    # GEMM fp16 con acumulador fp32. El default (reduccion fp16) desborda
    # el residual V4 a Inf/NaN alrededor del paso 350, justo al terminar el warmup.
    _mm = torch.backends.cuda.matmul
    if hasattr(_mm, 'allow_fp16_reduced_precision_reduction'):
        _mm.allow_fp16_reduced_precision_reduction = False
    print('GPUs:', [torch.cuda.get_device_name(i) for i in range(NGPU)])
else:
    print('Dispositivo: CPU  (el modo FULL es solo viable en GPU; usa FAST_MODE=True)')


## 2️⃣ Configuración central V4


In [ ]:
# ------------------------- MODO -----------------------------------------
FAST_MODE = False   # False => entrenamiento REAL ~20.4M / TinyStories completo / 512
SEED = 1234

# ------------------------- DATOS -----------------------------------------
SEQ_LEN = 512                 # contexto del modelo real (tokens GPT-2)
MAX_TRAIN_SEQS = None         # None = TODO el split de entrenamiento (~370k secuencias).
MAX_VALID_SEQS = 1_000        # secuencias de validacion (solo se usa en FAST)
TRAIN_FILE = 'tinystories_train.txt'
VALID_FILE = 'tinystories_valid.txt'
TRAIN_BYTES = 2_227_753_162   # TinyStoriesV2-GPT4-train.txt (~2.08 GiB)
VALID_BYTES = 22_502_601      # TinyStoriesV2-GPT4-valid.txt (~21.5 MiB)

# ------------------------- MODELO V4 (~20.4M en FULL) --------------------
MODEL_KW = dict(
    d_model=256, d_gate=32, d_ff=1024,
    num_cells=8, num_encoder_layers=2,
    num_consolidation_layers=9,   # L >= ceil(log2 512) => cobertura binaria completa
    num_candidates=4, candidate_aggregation='latent_fusion',
    synapse_rank=32, version='v4', offset_mode='resonant_multirate',
    gating_mode='dual', trace_tap=True, norm_type='rmsnorm', global_anchor=False,
)

# ------------------------- ENTRENAMIENTO ----------------------------------
BATCH_SIZE = 16 * max(1, NGPU)
EVAL_BATCH = 8
EPOCHS = 1
LR = 3e-4                 # 6e-4 + AMP fp16 reventaba a NaN ~paso 350 (tras el warmup)
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 500        # 200 era demasiado corto para AdamW(beta2=0.95) + pico de LR
LOG_EVERY = 50
EVAL_EVERY = 500
SAMPLE_EVERY = 500
EVAL_BATCHES = 25
RESUME = True
USE_AMP = (DEVICE == 'cuda')   # AMP fp16 (Tensor Cores). La CE va SIEMPRE en fp32.

if FAST_MODE:
    SEQ_LEN = 128
    MAX_TRAIN_SEQS = 256
    MAX_VALID_SEQS = 64
    MODEL_KW.update(d_model=128, d_gate=16, d_ff=512, num_cells=4,
                    num_encoder_layers=2, num_consolidation_layers=7,
                    num_candidates=2, synapse_rank=16)
    BATCH_SIZE, EVAL_BATCH, EPOCHS = 8, 8, 2
    WARMUP_STEPS = 10
    LOG_EVERY, EVAL_EVERY, SAMPLE_EVERY, EVAL_BATCHES = 10, 16, 16, 4

SAVE_DIR = ('/kaggle/working/engrama_v4_20m_gpt2'
            if os.path.isdir('/kaggle/working') else './engrama_v4_20m_gpt2')
random.seed(SEED)
torch.manual_seed(SEED)
print('Modo = %s | SEQ_LEN=%d | train_seqs=%s | batch=%d | AMP=%s | lr=%.1e | ckpt -> %s' % (
    'FAST' if FAST_MODE else 'FULL', SEQ_LEN,
    'TODO el split' if MAX_TRAIN_SEQS is None else '<= %s' % MAX_TRAIN_SEQS,
    BATCH_SIZE, USE_AMP, LR, SAVE_DIR))


## 3️⃣ Datos: TinyStories completo (descarga robusta)


In [ ]:
def iter_stories(path):
    buf = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.strip():
                buf.append(line.rstrip('\n'))
            elif buf:
                yield ' '.join(buf).strip()
                buf = []
    if buf:
        yield ' '.join(buf).strip()

def download_verified(url, path, expected_bytes, retries=8):
    import urllib.request
    done = os.path.getsize(path) if os.path.exists(path) else 0
    if done == expected_bytes:
        print('  %s: ya completo (%.0f MB)' % (path, done / 2**20))
        return path
    if 0 < done > expected_bytes:
        print('  %s: tamano incoherente, se redescarga desde cero' % path)
        os.remove(path)
        done = 0
    for attempt in range(1, retries + 1):
        mode = 'ab' if done else 'wb'
        headers = {'Range': 'bytes=%d-' % done} if done else {}
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp:
                total = done if resp.status == 206 else 0
                if resp.status == 200:
                    mode = 'wb'
                with open(path, mode) as f:
                    while True:
                        chunk = resp.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        f.write(chunk)
                        total += len(chunk)
                        if total % (256 * 1024 * 1024) < 4 * 1024 * 1024:
                            print('    %s: %.0f MB ...' % (path, total / 2**20))
            done = os.path.getsize(path)
            if done == expected_bytes:
                print('  %s: OK (%.0f MB verificados)' % (path, done / 2**20))
                return path
            print('  %s: incompleto (%d != %d); reintento ...' % (path, done, expected_bytes))
            if attempt >= retries:
                if os.path.exists(path):
                    os.remove(path)
                raise RuntimeError('Descarga de TinyStories fallida')
        except Exception as exc:
            done = os.path.getsize(path) if os.path.exists(path) else 0
            if attempt >= retries:
                if os.path.exists(path):
                    os.remove(path)
                raise RuntimeError('Descarga de TinyStories fallida') from exc
            print('  %s: %s; reintento %d/%d ...' % (path, type(exc).__name__, attempt, retries))
            time.sleep(min(30.0, 2 ** attempt))
    raise RuntimeError('Descarga de %s no verificada' % path)

def find_kaggle_file(basename, expected):
    for pat in ('/kaggle/input/*/' + basename, '/kaggle/input/**/' + basename):
        for hit in sorted(glob.glob(pat, recursive=True)):
            if os.path.getsize(hit) == expected:
                return hit
    return None

TRAIN_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-train.txt')
VALID_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-valid.txt')

FALLBACK_STORY = (
    'Once upon a time there was a little cat named Lily. Lily liked to play in the '
    'garden with her red ball. One day a dog named Tom came and they played together '
    'all day. At night Lily went home, ate her dinner and slept. The end.'
)

train_path = valid_path = None
if FAST_MODE:
    print('FAST_MODE: basta el split de validacion.')
    try:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
        train_path = valid_path
    except Exception as exc:
        with open('tinystories_synth.txt', 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(FALLBACK_STORY for _ in range(600)))
        train_path = valid_path = 'tinystories_synth.txt'
else:
    print('FULL: se requiere el dataset TinyStories COMPLETO (train ~2.08 GiB).')
    train_path = find_kaggle_file('TinyStoriesV2-GPT4-train.txt', TRAIN_BYTES)
    if train_path is None:
        train_path = download_verified(TRAIN_URL, TRAIN_FILE, TRAIN_BYTES)
    else:
        print('  train: dataset montado en Kaggle ->', train_path)
    valid_path = find_kaggle_file('TinyStoriesV2-GPT4-valid.txt', VALID_BYTES)
    if valid_path is None:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
    else:
        print('  valid: dataset montado en Kaggle ->', valid_path)

print('train:', train_path, '| valid:', valid_path)


## 4️⃣ Tokenizer GPT-2 (BPE, vocabulario 50,257)


In [ ]:
class GPT2Adapter:
    def __init__(self, hf_tok):
        self.tok = hf_tok
        eot = hf_tok.eos_token_id
        self.SPECIAL_TOKENS = {'<eos>': eot, '<bos>': eot, '<pad>': eot}
        self.vocab_size = len(hf_tok)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = list(self.tok.encode(text, add_special_tokens=False))
        if add_bos:
            ids = [self.SPECIAL_TOKENS['<bos>']] + ids
        if add_eos:
            ids = ids + [self.SPECIAL_TOKENS['<eos>']]
        return ids

    def decode(self, ids, skip_special_tokens=True):
        return self.tok.decode(list(ids), skip_special_tokens=skip_special_tokens)

    def encode_batch(self, texts):
        outs = self.tok(list(texts), add_special_tokens=False)['input_ids']
        return [list(ids) + [self.SPECIAL_TOKENS['<eos>']] for ids in outs]

tokenizer = None
_hf = None
try:
    os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
    from transformers import GPT2TokenizerFast
    _hf = GPT2TokenizerFast.from_pretrained('gpt2')
    if len(_hf) != 50257 or not _hf.encode('hello world', add_special_tokens=False):
        raise RuntimeError('GPT-2 incompleto')
    tokenizer = GPT2Adapter(_hf)
    print('Tokenizer: GPT-2 BPE, vocab =', tokenizer.vocab_size)
except Exception as exc:
    if not FAST_MODE:
        raise RuntimeError('El tokenizer GPT-2 es OBLIGATORIO en modo FULL') from exc
    from engrama import EngramaTokenizer
    sample = '\n'.join(s for _, s in zip(range(200), iter_stories(train_path)))
    tokenizer = EngramaTokenizer().fit_on_text(sample)
    print('Tokenizer: char-level fallback, vocab =', tokenizer.vocab_size)

VOCAB_SIZE = tokenizer.vocab_size
EOS_ID = tokenizer.SPECIAL_TOKENS['<eos>']


## 5️⃣ Tokenización en streaming → memmap


In [ ]:
class MemmapTokenWriter:
    def __init__(self, out_raw, initial_capacity):
        self.path = out_raw
        self.cap = max(1024, int(initial_capacity))
        self.mm = np.memmap(out_raw, dtype=np.int32, mode='w+', shape=(self.cap,))
        self.pos = 0

    def ensure(self, extra):
        if self.pos + extra <= self.cap:
            return
        new_cap = max(self.cap * 2, self.pos + extra)
        self.mm.flush()
        del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(new_cap * 4)
        self.mm = np.memmap(self.path, dtype=np.int32, mode='r+', shape=(new_cap,))
        self.cap = new_cap

    def extend(self, ids):
        self.ensure(len(ids))
        self.mm[self.pos:self.pos + len(ids)] = np.asarray(ids, dtype=np.int32)
        self.pos += len(ids)

    def finalize(self):
        self.mm.flush()
        n = int(self.pos)
        del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(n * 4)
        return np.memmap(self.path, dtype=np.int32, mode='r', shape=(n,))

def _flush_batch(writer, tokenizer, batch):
    if hasattr(tokenizer, 'encode_batch'):
        batch_ids = tokenizer.encode_batch(batch)
    else:
        batch_ids = [tokenizer.encode(t, add_eos=True) for t in batch]
    for ids in batch_ids:
        if ids:
            writer.extend(ids)

def stories_to_memmap(path, out_raw, tokenizer, batch_stories=256, max_ids=None):
    writer = MemmapTokenWriter(out_raw, initial_capacity=os.path.getsize(path) * 0.10)
    batch, n_stories = [], 0
    for story in iter_stories(path):
        batch.append(story)
        if len(batch) < batch_stories:
            continue
        _flush_batch(writer, tokenizer, batch)
        n_stories += len(batch)
        batch = []
        if n_stories % 25000 < batch_stories:
            print('  %7d cuentos | %10d tokens ...' % (n_stories, writer.pos))
        if max_ids and writer.pos >= max_ids:
            break
    if batch and not (max_ids and writer.pos >= max_ids):
        _flush_batch(writer, tokenizer, batch)
        n_stories += len(batch)
    print('  %7d cuentos | %10d tokens' % (n_stories, writer.pos))
    return writer.finalize()

t0 = time.time()
max_train_ids = MAX_TRAIN_SEQS * (SEQ_LEN + 1) if MAX_TRAIN_SEQS else None
max_valid_ids = MAX_VALID_SEQS * (SEQ_LEN + 1) if MAX_VALID_SEQS else None
print('Tokenizando split de entrenamiento ...')
train_mm = stories_to_memmap(train_path, 'tinystories_train.ids', tokenizer, max_ids=max_train_ids)
print('Tokenizando split de validacion ...')
valid_mm = stories_to_memmap(valid_path, 'tinystories_valid.ids', tokenizer, max_ids=max_valid_ids)
print('Tokenizacion lista en %ds' % int(time.time() - t0))


## 6️⃣ Dataset y DataLoaders optimizados (pin_memory + multi-workers)


In [ ]:
class TokenWindows(torch.utils.data.Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        self.n = len(tokens) // (seq_len + 1)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        start = i * (self.seq_len + 1)
        w = self.tokens[start:start + self.seq_len + 1]
        return torch.from_numpy(np.array(w[:-1], dtype=np.int64)), torch.from_numpy(np.array(w[1:], dtype=np.int64))

train_ds = TokenWindows(train_mm, SEQ_LEN)
valid_ds = TokenWindows(valid_mm, SEQ_LEN)
print('train: %s secuencias x %d tokens | valid: %s' % (format(len(train_ds), ','), SEQ_LEN, format(len(valid_ds), ',')))

rng = torch.Generator().manual_seed(SEED)
num_workers = 2 if DEVICE == 'cuda' else 0
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    generator=rng, num_workers=num_workers, pin_memory=(DEVICE == 'cuda'))
valid_dl = torch.utils.data.DataLoader(
    valid_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=0,
    pin_memory=(DEVICE == 'cuda'))


## 7️⃣ Construcción del modelo ENGRAMA V4 (~20.4M)


In [ ]:
config = EngramaConfig(vocab_size=VOCAB_SIZE, context_length=SEQ_LEN, **MODEL_KW)
torch.manual_seed(SEED)
model = EngramaModel(config).to(DEVICE)

n_params = model.num_parameters()
rf = config.receptive_field()
print('Parametros: %s  (~%.1fM)' % (format(n_params, ','), n_params / 1e6))
print('Campo receptivo: %s tokens | cubre contexto: %s' % (rf['max_reach'], rf['covers_context']))
print('Configuracion V4:', config.describe())

def _token_loss(logits, targets):
    """CE siempre en fp32. Bajo AMP, log_softmax fp16 sobre |V|=50,257 desborda a NaN."""
    return chunked_cross_entropy(logits.float(), targets)

class ShardLossModel(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x, y):
        logits = self.inner(x)
        # unsqueeze: DataParallel no sabe reunir escalares (warning + vector sucio)
        return _token_loss(logits, y).unsqueeze(0)

if NGPU > 1:
    train_model = nn.DataParallel(ShardLossModel(model), device_ids=list(range(NGPU)))
    print('Multi-GPU DataParallel sobre %d GPUs' % NGPU)
else:
    train_model = ShardLossModel(model)


## 8️⃣ Entrenamiento ultrarrápido con AMP FP16 (CE en fp32, pasos no-finitos se saltan)


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95), eps=1e-8)
# init_scale bajo: el default 2**16 hace overflow de grads en el residual V4 bajo fp16.
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP, init_scale=2 ** 12, growth_interval=2000)
os.makedirs(SAVE_DIR, exist_ok=True)

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

def _grads_finite():
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            return False
    return True

@torch.no_grad()
def evaluate(max_batches=EVAL_BATCHES):
    was_training = model.training
    model.eval()
    total, nb = 0.0, 0
    try:
        for i, (xb, yb) in enumerate(valid_dl):
            if i >= max_batches:
                break
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=torch.float16):
                logits = model(xb)
            loss = _token_loss(logits, yb)
            if not torch.isfinite(loss):
                continue
            total += loss.item() * xb.shape[0]
            nb += xb.shape[0]
    finally:
        if was_training:
            model.train()
    if nb == 0:
        return float('nan')
    return total / nb

def save_checkpoint(step, best):
    save_model(model, SAVE_DIR)
    torch.save({'optimizer': optimizer.state_dict(),
                'scaler': scaler.state_dict() if scaler is not None else None,
                'global_step': step, 'best_val': best},
               os.path.join(SAVE_DIR, 'trainer_state.pt'))
    if _hf is not None:
        _hf.save_pretrained(os.path.join(SAVE_DIR, 'tokenizer'))

generator = Generator(model, tokenizer)
history = []
best_val = float('inf')
start_step = 0
n_skipped = 0

if RESUME and os.path.exists(os.path.join(SAVE_DIR, 'model.pt')) \
        and os.path.exists(os.path.join(SAVE_DIR, 'trainer_state.pt')):
    try:
        model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'model.pt'), map_location=DEVICE))
        ck = torch.load(os.path.join(SAVE_DIR, 'trainer_state.pt'), map_location=DEVICE)
        optimizer.load_state_dict(ck['optimizer'])
        if scaler and ck.get('scaler'):
            scaler.load_state_dict(ck['scaler'])
        start_step = int(ck['global_step'])
        best_val = float(ck['best_val'])
        print('[resume] paso %d, best_val %.4f' % (start_step, best_val))
    except Exception as exc:
        print('[resume] fallo (%s); entrenando de cero' % exc)

TOTAL_STEPS = start_step + len(train_dl) * EPOCHS
print('Total pasos:', TOTAL_STEPS, '| Estimacion: ~0.2 s/paso -> %.1f horas' % (len(train_dl)*0.2/3600))

global_step = start_step
t0 = time.time()
model.train()
for epoch in range(EPOCHS):
    for xb, yb in train_dl:
        lr = lr_at(global_step)
        for gparam in optimizer.param_groups:
            gparam['lr'] = lr

        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=torch.float16):
            loss = train_model(xb, yb)
        if loss.dim() > 0:
            loss = loss.mean()

        if not torch.isfinite(loss):
            n_skipped += 1
            if n_skipped <= 8 or n_skipped % 50 == 0:
                print('  [skip] paso %d: loss no finita (saltados=%d, scaler=%.0f)' % (
                    global_step, n_skipped, scaler.get_scale() if USE_AMP else 1.0))
            global_step += 1
            continue

        took_step = False
        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if not _grads_finite():
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
                n_skipped += 1
                if n_skipped <= 8 or n_skipped % 50 == 0:
                    print('  [skip] paso %d: grads no finitos (saltados=%d, scaler=%.0f)' % (
                        global_step, n_skipped, scaler.get_scale()))
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                took_step = True
        else:
            loss.backward()
            if not _grads_finite():
                optimizer.zero_grad(set_to_none=True)
                n_skipped += 1
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                took_step = True

        if not took_step:
            global_step += 1
            continue

        loss_val = float(loss.detach())
        history.append((global_step, loss_val))

        if global_step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            s_per_step = elapsed / max(1, global_step - start_step + 1)
            extra = ' | skip %d' % n_skipped if n_skipped else ''
            print('paso %6d/%d | loss %.4f | lr %.2e | %.2fs/paso | %ds%s' % (
                global_step, TOTAL_STEPS, loss_val, lr, s_per_step, int(elapsed), extra))

        if (global_step + 1) % EVAL_EVERY == 0 or global_step + 1 == TOTAL_STEPS:
            val = evaluate()
            if math.isfinite(val):
                print('  [eval] paso %d: val_loss %.4f | val_ppl %.2f' % (
                    global_step + 1, val, math.exp(min(20.0, val))))
                if val < best_val:
                    best_val = val
                    torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model.pt'))
                    save_checkpoint(global_step + 1, best_val)
                    print('  [ckpt] mejor checkpoint guardado (val %.4f)' % best_val)
            else:
                print('  [eval] paso %d: val_loss nan (se omite ckpt)' % (global_step + 1,))

        if (global_step + 1) % SAMPLE_EVERY == 0:
            model.eval()
            try:
                print('  [muestra]', repr(generator.generate(
                    'Once upon a time', max_new_tokens=64, temperature=0.8, top_k=40, stop_at_eos=True)))
            except Exception as exc:
                print('  [muestra] omitida (%s)' % type(exc).__name__)
            model.train()

        global_step += 1

save_checkpoint(global_step, best_val)
print('Entrenamiento terminado en %ds | mejor val_loss %s | pasos saltados %d' % (
    int(time.time() - t0),
    '%.4f' % best_val if math.isfinite(best_val) else 'inf',
    n_skipped))


## 9️⃣ Inferencia y Muestras


In [ ]:
loaded_model, _ = load_model(SAVE_DIR, device=DEVICE)
weights = 'best_model.pt' if os.path.exists(os.path.join(SAVE_DIR, 'best_model.pt')) else 'model.pt'
loaded_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, weights), map_location=DEVICE))
loaded_model.eval()

gen = Generator(loaded_model, tokenizer)
for prompt in ['Once upon a time', 'One day, a little girl named Anna', 'Tom found a big red ball']:
    out = gen.generate(prompt, max_new_tokens=100, temperature=0.8, top_k=40, top_p=0.95, stop_at_eos=True)
    print('==>', prompt)
    print(out.replace('<|endoftext|>', '').strip()[:600], '\n')
